In [1]:
from stormpy import export_to_drn
%load_ext autoreload
%autoreload 2
import sys
import os
print(os.getpid())
sys.path.append('..')

from verimon.logger import setup_logging

setup_logging()

472307


In [2]:
from verimon import loaders

experiment_premise_a3 = ("../tests/premise/airportA-3.nm",  "DMAX=3,PMAX=3", 'P>0.5 [F<2 "crash" ]', "crash", 8)
experiment_premise_refuel = ("../tests/premise/refuel.nm", "N=12,ENERGY=50", "Pmax=? [F<=5 \"empty\"]", "empty", 14)
experiment_premise_evade = ("../tests/premise/evade-monitoring.nm", "N=5,RADIUS=3", "P>0.5 [F<=5 \"crash\"]", "crash", 10)

test_file, constants, spec, good_label, horizon = experiment_premise_a3

initial_observation, observations, mc, expr_manager = loaders.pomdp_to_mc(test_file, constants)

DEBUG:2024-11-20 10:00:12,089 - (0.00s) - loaders.py - Finished loading original pomdp with 18 observations 


DEBUG:2024-11-20 10:00:12,089 - (0.00s) - loaders.py - Finished assigning labels to states 
DEBUG:2024-11-20 10:00:12,090 - (0.00s) - loaders.py - Finished creating new transitions 
DEBUG:2024-11-20 10:00:12,090 - (0.00s) - loaders.py - transformed POMDP to stormpy DTMC 


In [3]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
import stormvogel

stormvogel.communication_server.enable_server = False

print(mc)
mc_sv = stormpy_to_stormvogel(mc)
# show(mc_sv)

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	45
Transitions: 	113
Reward Models:  none
State Labels: 	21 labels
   * {d: 2,pobs: 2,turn: 1} -> 2 item(s)
   * {d: 2,pobs: 2,turn: 0} -> 3 item(s)
   * {d: 0,pobs: 0,turn: 1} -> 2 item(s)
   * {d: 1,pobs: 1,turn: 1} -> 3 item(s)
   * {d: 1,pobs: 1,turn: 0} -> 3 item(s)
   * {d: 0,pobs: 2,turn: 1} -> 2 item(s)
   * crash -> 6 item(s)
   * {d: 1,pobs: 2,turn: 0} -> 3 item(s)
   * {d: 2,pobs: 1,turn: 0} -> 3 item(s)
   * {d: 2,pobs: 0,turn: 1} -> 2 item(s)
   * {d: 2,pobs: 1,turn: 1} -> 3 item(s)
   * {d: 1,pobs: 2,turn: 1} -> 2 item(s)
   * {d: 1,pobs: 0,turn: 0} -> 2 item(s)
   * {d: 0,pobs: 1,turn: 1} -> 3 item(s)
   * {d: 0,pobs: 1,turn: 0} -> 3 item(s)
   * deadlock -> 2 item(s)
   * {d: 2,pobs: 0,turn: 0} -> 2 item(s)
   * init -> 1 item(s)
   * {d: 0,pobs: 2,turn: 0} -> 3 item(s)
   * {d: 1,pobs: 0,turn: 1} -> 2 item(s)
   * {d: 0,pobs: 0,turn: 0} -> 2 item(s)
Choice Labels: 	none


In [ ]:
from aalpy import run_Lstar, Dfa
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.3
fp_slack = 0.5
fn_slack = 0.05
relative_error = 0.1

alphabet = list(observations)

filtering_sul = FilteringSUL(
    mc, 
    initial_observation, 
    alphabet, 
    spec, 
    threshold, 
    horizon
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc,
    threshold,
    fp_slack,
    fn_slack,
    horizon,
    spec,
    good_label,
    relative_error,
    False,
    expr_manager
)
learned_monitor: Dfa = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
) #type: ignore

AttributeError: 'stormpy.core.ExplicitQualitativeCheckResult' object has no attribute 'get_values'

In [ ]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import false_positive

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
# export_to_drn(stormvogel_to_stormpy(mon_cycl), "monitor.drn")
result_goal, trace, assignment, product = false_positive(mc, mon_cycl, horizon, expr_manager, options = {"good_spec": spec, "good_label": good_label})

DEBUG:2024-11-18 11:31:11,366 - (208.04s) - verify.py - Building model 
DEBUG:2024-11-18 11:31:11,369 - (0.00s) - verify.py - Unrolling done 
INFO:2024-11-18 11:31:11,383 - (0.01s) - generator.py - New good states become: [15, 19, 21, 36, 37, 38] 
DEBUG:2024-11-18 11:31:11,383 - (0.00s) - verify.py - Apply spec done 


DEBUG:2024-11-18 11:31:11,643 - (0.26s) - verify.py - creating product done 
DEBUG:2024-11-18 11:31:11,643 - (0.00s) - verify.py - Creating trace 
2024-11-18 11:31:11,846 - pomdp.py - constructed POMDP having 15 observations.
2024-11-18 11:31:11,852 - pomdp.py - unfolding 1-FSC template into POMDP...
2024-11-18 11:31:11,855 - pomdp.py - constructed quotient MDP having 374 states and 6757 actions.
2024-11-18 11:31:11,899 - statistic.py - synthesis initiated, design space: 1e16
2024-11-18 11:31:12,011 - synthesizer_ar.py - value 0.5003 achieved after 0.37 seconds
2024-11-18 11:31:12,097 - synthesizer_ar.py - value 0.5186 achieved after 0.45 seconds
2024-11-18 11:31:12,154 - synthesizer_ar.py - value 0.549 achieved after 0.51 seconds
2024-11-18 11:31:14,742 - synthesizer_ar.py - value 0.5571 achieved after 3.1 seconds
> progress 1.169%, elapsed 3 s, estimated 256 s, iters = {MDP: 605}, opt = 0.5571
2024-11-18 11:31:15,196 - synthesizer_ar.py - value 0.5882 achieved after 3.55 seconds
2024